In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, to_timestamp, explode

# 1. Setup Variables
catalog = "maritime_ais"
bronze_schema = "maritime_bronze"
silver_schema = "maritime_silver"
checkpoint_base = "abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/checkpoints/silver"

# 2. Define Upsert Logic with Spark Connect Compatibility & Error Logging
def upsert_to_silver(microBatchDF, batchId, table_name, primary_key):
    try:
        # Retrieve spark session safely from the batch DataFrame
        _spark = microBatchDF.sparkSession
        
        # Filter out rows with null primary keys to prevent merge crashes
        clean_df = microBatchDF.filter(col(primary_key).isNotNull())
        
        if clean_df.count() == 0:
            return

        if not _spark.catalog.tableExists(table_name):
            clean_df.write.format("delta").mode("overwrite").saveAsTable(table_name)
            return
            
        delta_table = DeltaTable.forName(_spark, table_name)
        merge_condition = f"target.{primary_key} = source.{primary_key}"
        
        delta_table.alias("target").merge(
            clean_df.alias("source"),
            merge_condition
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        
    except Exception as e:
        print(f"CRITICAL ERROR in batch {batchId}: {str(e)}")
        raise e

# 3. Process Sea State
print("Processing Silver Sea State...")
df_bronze_sea = spark.readStream.table(f"{catalog}.{bronze_schema}.sea_state")

df_exploded_sea = df_bronze_sea.select(explode(col("features")).alias("feature"))

df_silver_sea = df_exploded_sea.select(
    col("feature.properties.siteNumber").cast("string").alias("site_number"),
    col("feature.properties.siteName").alias("site_name"),
    col("feature.properties.siteType").alias("site_type"),
    col("feature.geometry.coordinates").getItem(0).cast("double").alias("longitude"),
    col("feature.geometry.coordinates").getItem(1).cast("double").alias("latitude"),
    col("feature.properties.seaState").alias("sea_state"),
    col("feature.properties.temperature").cast("double").alias("temperature"),
    to_timestamp(col("feature.properties.lastUpdate")).alias("last_update")
)

sea_table = f"{catalog}.{silver_schema}.sea_state"
query_sea = (
    df_silver_sea.writeStream
    .foreachBatch(lambda df, epoch_id: upsert_to_silver(df, epoch_id, sea_table, "site_number"))
    .option("checkpointLocation", f"{checkpoint_base}/sea_state")
    .trigger(availableNow=True)
    .start()
)
query_sea.processAllAvailable()
print(f"Completed Sea State Processing.")